# 01. 제약별 검증과 재귀 결정

## 학습 목표

- 후보 답을 여러 제약으로 나누어 검증합니다.
- 검증 결과와 궤적 복구 가능성으로 `ACCEPT`, `REFINE`, `RESTART`를 선택합니다.
- confidence 임계값의 효과를 관찰합니다.

이 notebook은 AREX 전체 모델이 아니라 논문의 외부 자기개선 루프를 작은 규칙 기반 예제로 재현합니다. 위에서 아래로 실행하면 되며 외부 패키지는 필요하지 않습니다.

In [ ]:
from dataclasses import dataclass
from typing import Callable

@dataclass(frozen=True)
class Constraint:
    name: str
    check: Callable[[dict], bool]
    follow_up: str

@dataclass(frozen=True)
class Audit:
    passed: tuple[str, ...]
    unresolved: tuple[str, ...]
    confidence: float

def audit_candidate(candidate: dict, constraints: list[Constraint]) -> Audit:
    """각 제약을 독립적으로 검사하고 통과 비율을 confidence로 사용합니다."""
    passed, unresolved = [], []
    for constraint in constraints:
        # 실제 시스템에서는 bool 대신 출처 품질과 모순 여부도 함께 검사해야 합니다.
        target = passed if constraint.check(candidate) else unresolved
        target.append(constraint.name)
    confidence = 100.0 * len(passed) / len(constraints)
    return Audit(tuple(passed), tuple(unresolved), confidence)

constraints = [
    Constraint("공식 출처", lambda c: c.get("official_source", False), "공식 문서를 찾는다"),
    Constraint("2026년 공개", lambda c: c.get("year") == 2026, "공개 연도를 확인한다"),
    Constraint("오픈 가중치", lambda c: c.get("open_weights", False), "모델 카드 라이선스를 확인한다"),
    Constraint("4B 이하", lambda c: c.get("parameters_b", 999) <= 4, "parameter 규모를 확인한다"),
]

candidate = {
    "name": "Toy-Research-4B",
    "official_source": True,
    "year": 2026,
    "open_weights": False,
    "parameters_b": 4,
}
audit = audit_candidate(candidate, constraints)
audit

In [ ]:
def decide(confidence: float, recoverable: bool, threshold: float = 85.0) -> str:
    """논문 Equation 10의 ACCEPT/REFINE/RESTART 규칙을 단순화합니다."""
    if confidence >= threshold:
        return "ACCEPT"
    return "REFINE" if recoverable else "RESTART"

def next_objective(audit: Audit, constraints: list[Constraint]) -> list[str]:
    """실패한 제약만 다음 라운드의 목표로 변환합니다."""
    unresolved = set(audit.unresolved)
    return [c.follow_up for c in constraints if c.name in unresolved]

decision = decide(audit.confidence, recoverable=True)
plan = next_objective(audit, constraints)
print("결정:", decision)
print("보존할 검증 결과:", audit.passed)
print("다음 연구 목표:", plan)

assert decision == "REFINE"
assert plan == ["모델 카드 라이선스를 확인한다"]

## 두 번째 라운드

첫 라운드에서 확인된 세 제약은 다시 검색하지 않습니다. 미해결이던 모델 카드만 확인해 후보를 갱신합니다. 이것이 단순 재시작과 `REFINE`의 차이입니다.

In [ ]:
refined_candidate = {**candidate, "open_weights": True}
second_audit = audit_candidate(refined_candidate, constraints)
print(second_audit)
print("두 번째 결정:", decide(second_audit.confidence, recoverable=True))

assert second_audit.confidence == 100.0
assert decide(second_audit.confidence, recoverable=True) == "ACCEPT"

# 같은 confidence라도 궤적이 복구 불가능하면 RESTART가 됩니다.
assert decide(50.0, recoverable=False) == "RESTART"

In [ ]:
# 임계값 민감도: confidence를 확률로 쓰기 전에 calibration 검사가 필요합니다.
for threshold in (60, 75, 85, 95):
    print(f"threshold={threshold:>2}: {decide(audit.confidence, True, threshold)}")

print("\n확장 과제: bool 제약을 출처 권위, 날짜, 모순을 포함한 0~1 점수로 바꿔 보세요.")